# LeetCode 3506 — Find Time Required to Eliminate Bacterial Strains bằng Python

- Link: https://leetcode.com/problems/find-time-required-to-eliminate-bacterial-strains/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: **Tham lam kiểu Huffman** (gộp ngược hai giá trị nhỏ nhất), **heap**, **binary search trên đáp án + bất đẳng thức Kraft**, hai hàng đợi

> **Ghi chú Premium:** đề bài và chữ ký `minEliminationTime(self, timeReq, splitTime)` lấy theo bản mô tả và lời giải tham khảo công khai trên doocs (leetcode.doocs.org) — nên đối chiếu lại nếu có tài khoản Premium.

Skeleton (Python3, theo doocs):

```python
class Solution:
    def minEliminationTime(self, timeReq: List[int], splitTime: int) -> int:
```

## 1. Mục tiêu học

1. Mô hình hóa quá trình "bạch cầu tự tách đôi" thành một **cây nhị phân**: mỗi lá là một chủng vi khuẩn, mỗi cạnh từ nút trong xuống là một lần tách tốn `splitTime`.
2. Viết chi phí: chủng `i` ở độ sâu `d_i` xong lúc `d_i · splitTime + timeReq[i]`; đáp án = **min trên mọi cây** của **max** các thời điểm đó.
3. Hiểu cách **nghĩ ngược**: thay vì tách bạch cầu, **gộp hai chủng** thành một "chủng ảo" có thời gian `splitTime + max(a, b)`; tham lam luôn gộp **hai giá trị nhỏ nhất** (giống Huffman).
4. Kiểm tra tính khả thi của một mốc thời gian `T` bằng **bất đẳng thức Kraft** `Σ 2^{-D_i} ≤ 1` ⇒ binary search.
5. Kiểm chứng bằng vét cạn mọi cây nhị phân trên dữ liệu nhỏ.

## 2. Đề bài & ví dụ

Có `n` chủng vi khuẩn; chủng `i` cần `timeReq[i]` đơn vị thời gian để tiêu diệt. Ban đầu có **một** bạch cầu.

- Một bạch cầu chỉ tiêu diệt được **một** chủng, sau đó kiệt sức.
- Một bạch cầu có thể **tách thành hai**, mất `splitTime`; hai bạch cầu mới làm việc song song.
- Mỗi chủng chỉ do một bạch cầu xử lý.

Trả về thời gian **ít nhất** để tiêu diệt hết.

**Ví dụ 1:** `timeReq = [10,4,5], splitTime = 2` → `12` (tách: một nhánh diệt 10 lúc `2 + 10 = 12`; nhánh kia tách tiếp, diệt 4 và 5 lúc `2+2+4`, `2+2+5`).

**Ví dụ 2:** `timeReq = [10,4], splitTime = 5` → `15`.

**Ràng buộc (theo doocs):** `2 ≤ n ≤ 10^5`, `1 ≤ timeReq[i] ≤ 10^9`, `1 ≤ splitTime ≤ 10^9`.

In [1]:
from typing import List
import heapq
examples = [([10, 4, 5], 2, 12), ([10, 4], 5, 15)]
for t, s, exp in examples:
    print(t, s, '-> kỳ vọng', exp)

[10, 4, 5] 2 -> kỳ vọng 12
[10, 4] 5 -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. Cây nhị phân của các lần tách

Mỗi lần tách tạo hai nhánh ⇒ kế hoạch tương ứng một **cây nhị phân đầy đủ** (mỗi nút trong có đúng 2 con) có `n` lá. Lá ở độ sâu `d` đã trải qua `d` lần tách.

### 3.2. Heap nhỏ nhất

In [2]:
h = [10, 4, 5]
heapq.heapify(h)
print(heapq.heappop(h), heapq.heappop(h), h)

4 5 [10]


### 3.3. Bất đẳng thức Kraft

Tồn tại cây nhị phân có các lá ở độ sâu `≤ D_1, …, D_n` (mỗi lá một chủng) **khi và chỉ khi** `Σ 2^{-D_i} ≤ 1`. Ví dụ `D = [1, 2, 2]`: `1/2 + 1/4 + 1/4 = 1` ⇒ được (đúng hình ví dụ 1).

In [3]:
from fractions import Fraction
def kraft(D):
    return sum(Fraction(1, 2 ** d) for d in D)
print(kraft([1, 2, 2]), kraft([1, 1, 2]))

1 5/4


### 3.4. Cách nghĩ ngược (gộp thay vì tách)

Hai lá anh em `a`, `b` có chung cha. Nhìn từ cha: cần `splitTime` để tách rồi mỗi bên làm việc song song ⇒ cha giống như một "chủng" cần `splitTime + max(a, b)`. Gộp lặp lại tới khi còn một chủng — giá trị cuối là thời gian của cả cây.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chủng **chậm** nên ở **gần gốc** (ít lần tách), chủng **nhanh** có thể ở sâu. Tính tay ví dụ 1 (`[10, 4, 5]`, `split = 2`):

- Gộp hai nhỏ nhất `4, 5` ⇒ `2 + 5 = 7`. Còn `[10, 7]`.
- Gộp `7, 10` ⇒ `2 + 10 = 12`. Đáp án **12**.

Nếu gộp `4` với `10` trước: `2 + 10 = 12`, rồi với `5`: `2 + 12 = 14` — tệ hơn.

In [4]:
def merge_order(times, split, order):
    """order: danh sách cặp chỉ số trong danh sách hiện tại để gộp lần lượt."""
    cur = list(times)
    for i, j in order:
        a, b = cur[i], cur[j]
        cur = [x for k, x in enumerate(cur) if k not in (i, j)] + [split + max(a, b)]
        print(f"gộp {a}, {b} -> {split + max(a, b)} | còn {cur}")
    return cur[0]

merge_order([10, 4, 5], 2, [(1, 2), (0, 1)])
print()
merge_order([10, 4, 5], 2, [(0, 1), (0, 1)])

gộp 4, 5 -> 7 | còn [10, 7]
gộp 10, 7 -> 12 | còn [12]

gộp 10, 4 -> 12 | còn [5, 12]
gộp 5, 12 -> 14 | còn [14]


14

## 5. Cách 1 — Brute Force: thử mọi cây nhị phân

`best(S)` cho một tập chủng `S`: nếu `|S| = 1` thì `t`; ngược lại thử mọi cách chia `S` thành hai phần khác rỗng `A, B`: `split + max(best(A), best(B))`. Ghi nhớ theo bitmask.

In [5]:
from functools import lru_cache

def min_time_brute(times, split):
    n = len(times)

    @lru_cache(maxsize=None)
    def best(mask):
        if mask & (mask - 1) == 0:
            return times[mask.bit_length() - 1]
        res = float('inf')
        sub = (mask - 1) & mask
        while sub:
            if sub < (mask ^ sub):           # mỗi cách chia chỉ xét một lần
                res = min(res, split + max(best(sub), best(mask ^ sub)))
            sub = (sub - 1) & mask
        return res

    return best((1 << n) - 1)

for t, s, exp in examples:
    print(min_time_brute(t, s), '| kỳ vọng', exp)

12 | kỳ vọng 12
15 | kỳ vọng 15


**Độ phức tạp:** `O(3^n)` — chỉ với `n ≤ 12`.

**Nút thắt:** cần biết cấu trúc của cây tối ưu thay vì thử mọi cây.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề tham lam (giống Huffman):** tồn tại cây tối ưu trong đó hai chủng **nhỏ nhất** là **anh em** ở độ sâu lớn nhất.

*Phác thảo:* xét lá sâu nhất trong một cây tối ưu, nó có anh em (cây đầy đủ). Đổi chỗ hai lá này với hai chủng nhỏ nhất: chủng nhỏ xuống sâu hơn, chủng lớn lên nông hơn ⇒ thời điểm hoàn thành của mọi lá không tăng (với `max`). Do đó có thể gộp hai chủng nhỏ nhất `a ≤ b` thành `split + b`, và bài toán thu nhỏ còn `n − 1` chủng.

**Thuật toán:** heap nhỏ nhất; lặp `n − 1` lần: lấy `a` (nhỏ nhất, bỏ đi), lấy `b`, đẩy `b + split`. Kết quả là phần tử cuối.

**Cách nhìn khác — binary search:** với mốc `T`, chủng `i` được phép ở độ sâu tối đa `D_i = ⌊(T − t_i)/split⌋` (cần `T ≥ t_i`). Khả thi ⇔ Kraft `Σ 2^{-D_i} ≤ 1`. Hàm khả thi đơn điệu theo `T` ⇒ tìm `T` nhỏ nhất.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Binary search + kiểm tra Kraft bằng "đếm chỗ trống theo tầng"

Kiểm tra Kraft chính xác không dùng phân số: duyệt các `D_i` theo thứ tự tăng; số chỗ trống ở tầng `d` là `slots`; khi xuống tầng sâu hơn `slots` nhân đôi (chặn ở `n` cho khỏi tràn); mỗi chủng có `D_i = d` chiếm một chỗ. Khả thi ⇔ không bao giờ thiếu chỗ.

In [6]:
def feasible(times_sorted_desc, split, T):
    n = len(times_sorted_desc)
    slots, depth = 1, 0                          # số chỗ trống ở tầng `depth`
    for t in times_sorted_desc:                  # t giảm dần ⇒ D tăng dần
        if t > T:
            return False
        D = (T - t) // split                     # độ sâu tối đa cho phép
        while depth < D and slots < n:           # đi xuống: mỗi chỗ thành 2 chỗ
            slots *= 2
            depth += 1
        if depth < D:
            depth = D                            # đã đủ chỗ (>= số chủng còn lại), không cần nhân thêm
        if slots == 0:
            return False
        slots -= 1                               # chủng này chiếm một chỗ ở tầng D
    return True

def min_time_binary(times, split):
    ts = sorted(times, reverse=True)
    lo, hi = max(times), max(times) + split * len(times).bit_length() + split
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(ts, split, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for t, s, exp in examples:
    print(min_time_binary(t, s), '| kỳ vọng', exp)

12 | kỳ vọng 12
15 | kỳ vọng 15


Chặn `slots ≤ n`: khi đã có từ `n` chỗ trở lên thì chắc chắn đủ cho mọi chủng còn lại (mỗi chủng chỉ cần một chỗ, các chỗ ở tầng nông có thể "đẩy xuống" thành nhiều chỗ sâu hơn). Cận trên `hi`: cây cân bằng sâu `⌈log2 n⌉` luôn khả thi với `T = max + split·⌈log2 n⌉`.

**Độ phức tạp:** `O(n log n)` sắp xếp + `O(n · log(range))` cho binary search.

### 7.2. Cách 3 — Heap kiểu Huffman (lời giải chính, theo doocs)

In [7]:
def min_time_heap(times, split):
    h = list(times)
    heapq.heapify(h)
    while len(h) > 1:
        heapq.heappop(h)                          # a nhỏ nhất: max(a, b) = b
        heapq.heappush(h, heapq.heappop(h) + split)
    return h[0]

for t, s, exp in examples:
    print(min_time_heap(t, s), '| kỳ vọng', exp)

12 | kỳ vọng 12
15 | kỳ vọng 15


**Độ phức tạp:** `O(n log n)`.

### 7.3. Cách 4 — Hai hàng đợi (Huffman tuyến tính sau khi sắp xếp)

Các giá trị mới tạo ra `b + split` **không giảm** theo thời gian (vì `b` là phần tử nhỏ thứ hai, và các lần sau lấy phần tử lớn hơn hoặc bằng). Nên dùng hai hàng đợi: `A` = mảng đã sắp, `B` = các giá trị mới (tự nhiên tăng dần); mỗi lần lấy nhỏ nhất từ đầu hai hàng đợi ⇒ `O(n)` sau khi sắp xếp.

In [8]:
from collections import deque

def min_time_two_queues(times, split):
    A = deque(sorted(times))
    B = deque()

    def pop_min():
        if not B or (A and A[0] <= B[0]):
            return A.popleft()
        return B.popleft()

    for _ in range(len(times) - 1):
        pop_min()
        b = pop_min()
        B.append(b + split)
    return (A or B)[0]

for t, s, exp in examples:
    print(min_time_two_queues(t, s), '| kỳ vọng', exp)

12 | kỳ vọng 12
15 | kỳ vọng 15


## 8. TRACE từng bước

Trace cách 3 với `timeReq = [3, 1, 4, 1, 5, 9]`, `splitTime = 2`: in heap trước mỗi lần gộp.

In [9]:
def trace(times, split):
    h = list(times)
    heapq.heapify(h)
    step = 0
    while len(h) > 1:
        a = heapq.heappop(h)
        b = heapq.heappop(h)
        heapq.heappush(h, b + split)
        step += 1
        print(f"bước {step}: gộp a={a}, b={b} -> {split} + max = {b + split} | heap {sorted(h)}")
    print('đáp án =', h[0])

trace([3, 1, 4, 1, 5, 9], 2)
print('vét cạn:', min_time_brute([3, 1, 4, 1, 5, 9], 2))

bước 1: gộp a=1, b=1 -> 2 + max = 3 | heap [3, 3, 4, 5, 9]
bước 2: gộp a=3, b=3 -> 2 + max = 5 | heap [4, 5, 5, 9]
bước 3: gộp a=4, b=5 -> 2 + max = 7 | heap [5, 7, 9]
bước 4: gộp a=5, b=7 -> 2 + max = 9 | heap [9, 9]
bước 5: gộp a=9, b=9 -> 2 + max = 11 | heap [11]
đáp án = 11
vét cạn: 11


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minEliminationTime(self, timeReq: List[int], splitTime: int) -> int:
        h = list(timeReq)
        heapq.heapify(h)
        while len(h) > 1:
            heapq.heappop(h)
            heapq.heappush(h, heapq.heappop(h) + splitTime)
        return h[0]

for t, s, exp in examples:
    got = Solution().minEliminationTime(t, s)
    print(got, 'OK' if got == exp else 'SAI')

12 OK
15 OK


## 10. Vì sao thuật toán đúng

1. **Mô hình cây:** mọi kế hoạch hợp lệ ↔ một cây nhị phân đầy đủ với `n` lá (mỗi lá một chủng); thời điểm xong của lá `i` là `d_i · split + t_i` (các lần tách trên đường từ gốc xảy ra tuần tự, các nhánh khác chạy song song). Đáp án = `min_cây max_i (d_i·split + t_i)`.
2. **Giá trị của cây theo đệ quy:** `val(lá) = t`, `val(nút) = split + max(val(con trái), val(con phải))` — đúng bằng max ở trên.
3. **Bổ đề trao đổi:** trong một cây tối ưu, lấy hai lá anh em sâu nhất `x, y`. Đổi nhãn sao cho chúng mang hai giá trị nhỏ nhất `a ≤ b`: lá cũ mang giá trị lớn hơn được đưa lên vị trí nông hơn hoặc bằng ⇒ không làm tăng `max`. Vậy tồn tại cây tối ưu có `a, b` là anh em.
4. **Quy nạp:** thay `a, b` bằng một lá giá trị `split + b`; cây tối ưu cho bài toán mới mở rộng thành cây tối ưu cho bài toán cũ (và ngược lại) với cùng giá trị ⇒ lặp tham lam cho đáp án đúng.
5. **Binary search (cách 2):** Kraft là điều kiện cần và đủ để có cây với độ sâu lá `≤ D_i`; `D_i` không giảm theo `T` ⇒ khả thi đơn điệu.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn cây (bitmask) | `O(3^n)` | `O(2^n)` |
| 2. Binary search + Kraft | `O(n log n + n log R)` | `O(n)` |
| 3. Heap kiểu Huffman | `O(n log n)` | `O(n)` |
| 4. Hai hàng đợi | `O(n log n)` (sắp xếp) + `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'binary': min_time_binary, 'heap': min_time_heap, 'two_queues': min_time_two_queues,
         'submit': lambda t, s: Solution().minEliminationTime(t, s)}
edge = [([1, 1], 1, 2), ([5, 5, 5, 5], 1, 7), ([100, 1, 1, 1], 1, 101), ([1] * 8, 10, 31), ([10**9, 10**9], 10**9, 2 * 10**9)]
for t, s, exp in examples + edge:
    if len(t) <= 10:
        assert min_time_brute(t, s) == exp, (t, s)
    for name, f in impls.items():
        got = f(t, s)
        assert got == exp, (name, t, s, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
random.seed(3506)
cnt = 0
for _ in range(400):
    n = random.randint(2, 8)
    t = [random.randint(1, 20) for _ in range(n)]
    s = random.randint(1, 10)
    exp = min_time_brute(t, s)
    for name, f in impls.items():
        assert f(t, s) == exp, (name, t, s)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 400 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 10^5`)

In [13]:
import time
random.seed(5)
for name, t, s in [('ngẫu nhiên', [random.randint(1, 10**9) for _ in range(10**5)], random.randint(1, 10**9)),
                   ('toàn 1, split lớn', [1] * 10**5, 10**9),
                   ('split nhỏ', [random.randint(1, 10**9) for _ in range(10**5)], 1)]:
    t0 = time.time(); a = Solution().minEliminationTime(t, s); t1 = time.time() - t0
    t0 = time.time(); b = min_time_binary(t, s); t2 = time.time() - t0
    c = min_time_two_queues(t, s)
    assert a == b == c
    print(f"{name}: {a} | heap {t1:.2f}s | binary search {t2:.2f}s")

ngẫu nhiên: 7465734651 | heap 0.09s | binary search 0.40s


toàn 1, split lớn: 17000000001 | heap 0.06s | binary search 0.30s


split nhỏ: 999998091 | heap 0.13s | binary search 0.10s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gộp hai giá trị **lớn nhất** (hoặc tách đều như cây cân bằng)

In [14]:
def balanced_tree(times, split):
    import math
    return max(times) + split * math.ceil(math.log2(len(times)))     # SAI: luôn cây cân bằng

print('sai :', balanced_tree([100, 1, 1, 1], 1), '| đúng:', Solution().minEliminationTime([100, 1, 1, 1], 1))

sai : 102 | đúng: 101


**Vì sao sai:** chủng chậm (100) nên đứng ngay sau **một** lần tách; các chủng nhanh có thể ở sâu.

### Lỗi 2 — Cộng `split` vào **phần tử nhỏ** (max sai)

In [15]:
def wrong_add_to_min(times, split):
    h = list(times)
    heapq.heapify(h)
    while len(h) > 1:
        a = heapq.heappop(h)
        heapq.heappop(h)
        heapq.heappush(h, a + split)           # SAI: phải là max(a, b) + split
    return h[0]

print('sai :', wrong_add_to_min([10, 4], 5), '| đúng:', Solution().minEliminationTime([10, 4], 5))

sai : 9 | đúng: 15


**Vì sao sai:** sau khi tách, hai bạch cầu làm song song ⇒ thời gian của nút cha là `split + max`, không phải `split + min`.

### Lỗi 3 — Cộng `split` cho **mỗi** chủng (coi các lần tách là tuần tự toàn cục)

In [16]:
def wrong_sequential(times, split):
    return (len(times) - 1) * split + max(times)          # SAI

print('sai :', wrong_sequential([10, 4, 5], 2), '| đúng:', Solution().minEliminationTime([10, 4, 5], 2))
print('sai :', wrong_sequential([1] * 8, 10), '| đúng:', Solution().minEliminationTime([1] * 8, 10))

sai : 14 | đúng: 12
sai : 71 | đúng: 31


**Vì sao sai:** các bạch cầu sau khi tách có thể **tách song song**; chỉ các lần tách trên **cùng một nhánh** mới cộng dồn.

### Lỗi 4 — Kiểm tra Kraft bằng số thực

In [17]:
D = [1] + [60] * 4
print('số thực:', sum(2.0 ** -d for d in D) <= 1, '| chính xác:', sum(Fraction(1, 2 ** d) for d in D) <= 1)
D2 = [1, 1, 60]
print('số thực:', sum(2.0 ** -d for d in D2) <= 1, '| chính xác:', sum(Fraction(1, 2 ** d) for d in D2) <= 1)

số thực: True | chính xác: True
số thực: True | chính xác: False


**Vì sao sai:** `1/2 + 1/2 + 2^{-60}` lớn hơn 1 nhưng số thực làm tròn thành 1 ⇒ tưởng khả thi. Dùng phép đếm chỗ trống bằng số nguyên (cách 2) hoặc `Fraction`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Một tác nhân tự nhân đôi tốn thời gian, rồi làm song song" ⇒ **cây nhị phân**; chi phí kiểu `split + max(con)`.
- "Tối ưu cây nhị phân với chi phí gộp tăng theo các con" ⇒ thử **tham lam Huffman** (gộp hai nhỏ nhất) và chứng minh bằng trao đổi.
- Kiểm tra "có cây với độ sâu lá bị chặn" ⇒ **bất đẳng thức Kraft**.
- Giá trị mới tạo ra đơn điệu ⇒ hai hàng đợi thay heap.

## 15. Biến thể của bài

1. Mỗi lần tách tạo **ba** bạch cầu ⇒ cây tam phân; tham lam gộp ba nhỏ nhất (cần thêm chủng "giả" thời gian 0 để số lá hợp lệ).
2. Chi phí gộp là **tổng** thay vì max (Huffman cổ điển, LC 1167 *Minimum Cost to Connect Sticks*).
3. Thời gian tách khác nhau theo tầng ⇒ tham lam vẫn đúng nếu chi phí tầng không đổi theo nhánh.
4. Bài liên quan: LC 1199 *Minimum Time to Build Blocks* — **cùng bài toán** (khối = chủng, công nhân = bạch cầu).

In [18]:
# Biến thể 4: LC 1199 có cùng lời giải
def min_build_time(blocks, split):
    return Solution().minEliminationTime(blocks, split) if len(blocks) > 1 else blocks[0]

print(min_build_time([1], 1), min_build_time([1, 2], 5), min_build_time([1, 2, 3], 1))   # 1, 7, 4 (ví dụ LC 1199)

1 7 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Vét cạn cây | `O(3^n)` | Oracle |
| 2 | Binary search + Kraft | `O(n log n + n log R)` | Khi khó thấy tham lam |
| 3 | Heap Huffman | `O(n log n)` | **Nộp bài** |
| 4 | Hai hàng đợi | `O(n)` sau sắp xếp | Khi đã có mảng sắp xếp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minEliminationTime([1, 2, 3], 1)`.

**Bài 2 (tính tay):** `minEliminationTime([8, 8, 8, 8], 3)`.

**Bài 3 (điền TODO):** viết `depths_from_heap(times, split)` trả về độ sâu của từng chủng trong cây tham lam.

**Bài 4 (tự viết lại):** biến thể 1 — mỗi lần tách tạo **ba** bạch cầu.

**Bài 5 (mini challenge):** kiểm tra bằng code rằng đáp án của cách 3 bằng `min T` sao cho Kraft khả thi (tức cách 2) trên 1000 bộ ngẫu nhiên.

In [19]:
def depths_from_heap_todo(times, split):
    # TODO: mỗi phần tử heap giữ (giá trị, danh sách chỉ số lá); khi gộp, cộng 1 độ sâu cho mọi lá của hai nhóm
    raise NotImplementedError

def ternary_todo(times, split):
    # TODO
    raise NotImplementedError

def cross_check_todo():
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Gộp 1, 2 ⇒ 3. (2) Gộp 3, 3 ⇒ 4. (3) `4`.

**Bài 2.** (1) Cây cân bằng 2 tầng. (2) `8 + 2·3`. (3) `14`.

**Bài 3.** (1) Phần tử heap: `(giá trị, id, danh sách lá)`. (2) Khi gộp, mọi lá của hai nhóm thêm 1 độ sâu. (3) Kiểm tra `max(d_i·split + t_i)` bằng đáp án.

**Bài 4.** (1) Nếu `(n − 1) mod 2 ≠ 0` thì thêm chủng giả thời gian `0` cho đủ (mỗi lần gộp giảm 2). (2) Gộp ba nhỏ nhất `a ≤ b ≤ c` ⇒ `split + c`. (3) Vét cạn cây tam phân nhỏ để kiểm.

**Bài 5.** (1) Dùng `min_time_binary` và `min_time_heap`. (2) Sinh dữ liệu ngẫu nhiên. (3) So sánh.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
a1 = Solution().minEliminationTime([1, 2, 3], 1)
a2 = Solution().minEliminationTime([8, 8, 8, 8], 3)
assert a1 == 4 == min_time_brute([1, 2, 3], 1) and a2 == 14 == min_time_brute([8, 8, 8, 8], 3)
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 4 | Bài 2: 14


In [21]:
# Bài 3: độ sâu của từng chủng trong cây tham lam
def depths_from_heap(times, split):
    h = [(t, i, [i]) for i, t in enumerate(times)]
    heapq.heapify(h)
    depth = [0] * len(times)
    uid = len(times)
    while len(h) > 1:
        a = heapq.heappop(h)
        b = heapq.heappop(h)
        for leaf in a[2] + b[2]:
            depth[leaf] += 1
        heapq.heappush(h, (b[0] + split, uid, a[2] + b[2]))
        uid += 1
    return depth

random.seed(3)
for _ in range(200):
    t = [random.randint(1, 30) for _ in range(random.randint(2, 9))]
    s = random.randint(1, 5)
    d = depths_from_heap(t, s)
    assert max(di * s + ti for di, ti in zip(d, t)) == Solution().minEliminationTime(t, s)
    assert sum(Fraction(1, 2 ** di) for di in d) == 1          # cây đầy đủ
print('Bài 3: ví dụ 1 ->', depths_from_heap([10, 4, 5], 2))

Bài 3: ví dụ 1 -> [1, 2, 2]


In [22]:
# Bài 4: tách thành ba
def ternary(times, split):
    h = list(times)
    while (len(h) - 1) % 2:
        h.append(0)                      # chủng giả
    heapq.heapify(h)
    while len(h) > 1:
        heapq.heappop(h); heapq.heappop(h)
        heapq.heappush(h, heapq.heappop(h) + split)
    return h[0]

def ternary_brute(times, split):
    n = len(times)
    @lru_cache(maxsize=None)
    def best(mask):
        if mask & (mask - 1) == 0:
            return times[mask.bit_length() - 1]
        res = float('inf')
        # chia thành 2 hoặc 3 phần khác rỗng (một bạch cầu con có thể không dùng)
        s1 = (mask - 1) & mask
        while s1:
            rest = mask ^ s1
            res = min(res, split + max(best(s1), best(rest)))
            s2 = (rest - 1) & rest
            while s2:
                s3 = rest ^ s2
                res = min(res, split + max(best(s1), best(s2), best(s3)))
                s2 = (s2 - 1) & rest
            s1 = (s1 - 1) & mask
        return res
    return best((1 << n) - 1)

random.seed(4)
for _ in range(150):
    t = [random.randint(1, 20) for _ in range(random.randint(2, 7))]
    s = random.randint(1, 6)
    assert ternary(t, s) == ternary_brute(t, s), (t, s)
print('Bài 4: tách ba khớp vét cạn; ví dụ', ternary([10, 4, 5, 3], 2))

Bài 4: tách ba khớp vét cạn; ví dụ 12


In [23]:
# Bài 5
random.seed(5)
for _ in range(1000):
    t = [random.randint(1, 50) for _ in range(random.randint(2, 30))]
    s = random.randint(1, 20)
    assert min_time_heap(t, s) == min_time_binary(t, s)
print('Bài 5: Huffman và binary search + Kraft khớp trên 1000 bộ ngẫu nhiên')

Bài 5: Huffman và binary search + Kraft khớp trên 1000 bộ ngẫu nhiên


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
heap = timeReq
while len(heap) > 1:
    pop (nhỏ nhất, bỏ)
    push(pop() + splitTime)       # max(a, b) + split = b + split
return heap[0]
# kiểm tra: Kraft Σ 2^(-⌊(T - t_i)/split⌋) <= 1 (đếm chỗ trống bằng số nguyên)
```

**Checklist:**

- [ ] Gộp **hai nhỏ nhất**, giá trị mới `split + max`?
- [ ] Không cộng `split` tuần tự cho mọi chủng?
- [ ] Dùng số nguyên (Python không tràn; ngôn ngữ khác dùng `long`)?
- [ ] Đối chiếu với vét cạn cây trên dữ liệu nhỏ?
- [ ] Ghi chú chữ ký theo doocs (bài Premium)?